# Label clean for Multi-label  Pyfume dataset

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
'''
Plotting libraries
'''
import pandas as pd
import matplotlib.cm as cm
#from matplotlib import pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.ticker import (MultipleLocator, AutoMinorLocator)
fontsize=30
from numpy.ma.core import MAError
from tqdm import tqdm
'''
What we'll need for analysis, clustering, etc.
'''
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import silhouette_samples, silhouette_score
from sklearn.cluster import KMeans
from sklearn import datasets, decomposition
from sklearn.manifold import TSNE
from sklearn.feature_selection import RFE
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import make_classification
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

%config Completer.use_jedi = False

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split


import torch
from torch import Tensor
from torch.nn import Linear
from torch.nn import Sigmoid
from torch.nn import Module
from torch.optim import Adam
from torch.nn import MSELoss
from torch.nn.init import xavier_uniform_


In [3]:
!pip install iterative-stratification
!pip install nltk
!pip install fuzzywuzzy

In [4]:
import nltk
import sklearn

print('The nltk version is {}.'.format(nltk.__version__))
print('The scikit-learn version is {}.'.format(sklearn.__version__))

The nltk version is 3.8.1.
The scikit-learn version is 1.2.2.


In [5]:
import matplotlib
print('The nltk version is {}.'.format(matplotlib.__version__))

The nltk version is 3.7.1.


In [6]:
## unique MOL
uni_t = pd.read_csv('/content/drive/MyDrive/pgml/Data analysis/clean_1810/com_label_0711.csv')
uni_t

,Unnamed: 0,name,IsomericSMILES,CID,acid,aldehidic,aldehydic,almond,almondy,ambre,...,"alliaceous (onion, garlic)",blossom,raspberry,fennel,iris,corn,balsam,mango,lavender,beer
0,0,abietic acid,CC(C)C1=CC2=CC[C@@H]3[C@@]([C@H]2CC1)(CCC[C@@]...,10569.0,0.0,0.0,0.0,0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,dihydroabietyl alcohol,CC(C)C1CCC2C(=C1)CCC3C2(CCCC3(C)CO)C,101602.0,0.0,0.0,0.0,0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,benzyl methoxyethyl acetal,CC(OCCOC)OCC1=CC=CC=C1,22235151.0,0.0,0.0,0.0,0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,"2-methyl-1,3-dioxocane",CC1OCCCCCO1,68074566.0,0.0,0.0,0.0,0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,"2,2-diethoxypropane",CCOC(C)(C)OCC,31361.0,0.0,0.0,0.0,0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3740,3742,sodium alginate,[C@@H]1([C@H]([C@H](OC([C@@H]1O)O)C(=O)[O-])O)...,23693089.0,0.0,0.0,0.0,0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3741,3743,galactoarabinan,CC1C(C(C(C(O1)CO)O)OC2C(C(C(C(O2)COC3C(C(C(CO3...,24847856.0,0.0,0.0,0.0,0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3742,3744,67952-65-2,CC1=CN=C(C(=N1)C)SC,55251157.0,0.0,0.0,0.0,1,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3743,3745,"pyroligneous acids, reaction products with et ...",CC.OO,91001317.0,0.0,0.0,0.0,0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
## Common mol
com = pd.read_csv('/content/drive/MyDrive/pgml/Data analysis/common/all6.csv')
com

,name,IsomericSMILES,CID,'acid','alcohol','alcoholic','aldehidic','aldehydic','alliaceous (onion,'alliaceous',...,'walnut','warm','waxy','whiskey','wine-like','winelike','winey','wood','woody',garlic)'
0,acetaldehyde,CC=O,177.0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,13002-09-0,CC(C)CCOC(C)OCCC(C)C,83036.0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,acetal,CCOC(C)OCC,7765.0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,63449-64-9,CC/C=C\CCOC(OCC/C=C\CC)C,5367767.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,"1,1-dipropoxyethane",CCCOC(C)OCCC,66929.0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1506,l-valine,CC(C)[C@@H](C(=O)O)N,6287.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1507,diethyl phthalate,CCOC(=O)C1=CC=CC=C1C(=O)OCC,6781.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1508,phenethyl anthranilate,C1=CC=C(C=C1)CCOC(=O)C2=CC=CC=C2N,8615.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1509,pyrrolidine,C1CCNC1,31268.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


#### Findout same meanings labels

In [8]:
# Remove single quotes from column headers
label=com.copy()
label.rename(columns={col: col.replace("'", "") for col in label.columns}, inplace=True)
label_com=label.drop(columns=['name','IsomericSMILES'])
label_com

,CID,acid,alcohol,alcoholic,aldehidic,aldehydic,alliaceous (onion,alliaceous,almond,almondy,...,walnut,warm,waxy,whiskey,wine-like,winelike,winey,wood,woody,garlic)
0,177.0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,83036.0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,7765.0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,5367767.0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,66929.0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1506,6287.0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1507,6781.0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1508,8615.0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1509,31268.0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
uni_t_f=uni_t.drop(columns=['name','Unnamed: 0','IsomericSMILES'])
uni_t_f

,CID,acid,aldehidic,aldehydic,almond,almondy,ambre,animal,anisic,apple,...,"alliaceous (onion, garlic)",blossom,raspberry,fennel,iris,corn,balsam,mango,lavender,beer
0,10569.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,101602.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,22235151.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,68074566.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,31361.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3740,23693089.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3741,24847856.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3742,55251157.0,0.0,0.0,0.0,1,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3743,91001317.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
label_all=pd.concat([uni_t_f, label_com],ignore_index=True).fillna(0)
label_all

,CID,acid,aldehidic,aldehydic,almond,almondy,ambre,animal,anisic,apple,...,raspberry,fennel,iris,corn,balsam,mango,lavender,beer,alliaceous (onion,garlic)
0,10569.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,101602.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,22235151.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,68074566.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,31361.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5251,6287.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5252,6781.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5253,8615.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5254,31268.0,0.0,0.0,0.0,0,0.0,0.0,1,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
label_all.to_csv('/content/drive/MyDrive/pgml/Data analysis/common/update17janlabel_all.csv',index=False)

### Original label data

In [11]:
#label_all=pd.read_csv('/content/drive/MyDrive/pgml/Data analysis/common/label_all.csv')
label_all=pd.read_csv('/content/drive/MyDrive/pgml/Data analysis/common/update17jan/label_all.csv')
label_all

,CID,acid,aldehidic,aldehydic,almond,almondy,ambre,animal,anisic,apple,...,raspberry,fennel,iris,corn,balsam,mango,lavender,beer,alliaceous (onion,garlic)
0,10569.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,101602.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,22235151.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,68074566.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,31361.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5251,6287.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5252,6781.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5253,8615.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5254,31268.0,0.0,0.0,0.0,0,0.0,0.0,1,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Final number of labels

In [12]:
label_all.loc[:, (label_all != 0).any(axis=0)]  # https://stackoverflow.com/questions/21164910/how-do-i-delete-a-column-that-contains-only-zeros-in-pandas

,CID,acid,aldehidic,aldehydic,almond,almondy,ambre,animal,anisic,apple,...,raspberry,fennel,iris,corn,balsam,mango,lavender,beer,alliaceous (onion,garlic)
0,10569.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,101602.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,22235151.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,68074566.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,31361.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5251,6287.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5252,6781.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5253,8615.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5254,31268.0,0.0,0.0,0.0,0,0.0,0.0,1,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
label_all=label_all.drop(columns=['CID'])

## Use PorterStemmer and  fuzzywuzzy to find same semantic stem

In [16]:
## PorterStemmer
import nltk
nltk.download('punkt')
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# Initialize Porter Stemmer
porter = PorterStemmer()
headers = label_all.columns.tolist()

# Tokenize and apply Porter stemming
stemmed_headers = {}
for header in headers:
    words = word_tokenize(header)
    stemmed_words = [porter.stem(word) for word in words]
    stemmed_key = " ".join(stemmed_words)

    if stemmed_key in stemmed_headers:
        stemmed_headers[stemmed_key].append(header)
    else:
        stemmed_headers[stemmed_key] = [header]

# Print headers with the same core meaning
result_df=[]
for key, value in stemmed_headers.items():
    if len(value) > 1:
        print(f"Headers with core meaning '{key}': {', '.join(value)}")
        result_df.append(value)
result_df2 = pd.DataFrame(result_df)
result_df2

Headers with core meaning 'anis': anisic, anise
Headers with core meaning 'balsam': balsamic, balsam
Headers with core meaning 'camphor': camphor, camphoreous
Headers with core meaning 'caramel': caramelic, caramellic, caramel
Headers with core meaning 'jasmin': jasmin, jasmine
Headers with core meaning 'alcohol': alcoholic, alcohol
Headers with core meaning 'coumarin': coumarinic, coumarin


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


,0,1,2
0,anisic,anise,None
1,balsamic,balsam,None
2,camphor,camphoreous,None
3,caramelic,caramellic,caramel
4,jasmin,jasmine,None
5,alcoholic,alcohol,None
6,coumarinic,coumarin,None


In [17]:
## fuzzywuzzy
from fuzzywuzzy import fuzz
# Function to find headers with the same core meaning
def find_similar_headers(dataframe):
    similar_headers = []
    checked_headers = set()
    for column1 in dataframe.columns:
        if column1 not in checked_headers:
            similar_headers_group = [column1]
            for column2 in dataframe.columns:
                if column1 != column2 and column2 not in checked_headers and fuzz.ratio(column1, column2) > 80:
                    similar_headers_group.append(column2)
                    checked_headers.add(column2)
            checked_headers.add(column1)
            if len(similar_headers_group) > 1:
                similar_headers.append(similar_headers_group)
    return similar_headers

# Find headers with the same core meaning
headers_with_same_core_meaning = find_similar_headers(label_all)

# Print headers with the same core meaning
for headers in headers_with_same_core_meaning:
    print(f"Headers with the same core meaning: {', '.join(headers)}")
result_dff = pd.DataFrame(headers_with_same_core_meaning)

# Display the result as a table
result_dff

Headers with the same core meaning: aldehidic, aldehydic
Headers with the same core meaning: almond, almondy
Headers with the same core meaning: balsamic, basalmic, balsam
Headers with the same core meaning: buttery, butter
Headers with the same core meaning: camphoraceous, camphoreous
Headers with the same core meaning: caramelic, caramellic, caramel
Headers with the same core meaning: citrus, citrusy
Headers with the same core meaning: coco, cocoa
Headers with the same core meaning: ehtereal, ethereal
Headers with the same core meaning: floral, foral
Headers with the same core meaning: fruit, fruity
Headers with the same core meaning: gassy, grassy
Headers with the same core meaning: green, green,
Headers with the same core meaning: herbal, herba-
Headers with the same core meaning: jasmin, jasmine
Headers with the same core meaning: leather, leathery
Headers with the same core meaning: lilac, lillac
Headers with the same core meaning: mint, minty
Headers with the same core meaning: 

,0,1,2
0,aldehidic,aldehydic,None
1,almond,almondy,None
2,balsamic,basalmic,balsam
3,buttery,butter,None
4,camphoraceous,camphoreous,None
5,caramelic,caramellic,caramel
6,citrus,citrusy,None
7,coco,cocoa,None
8,ehtereal,ethereal,None
9,floral,foral,None


## Combine same core meaning

In [18]:
import pandas as pd

# Creating a sample DataFrame
data = {'A': [1, 2, 3], 'B': [4, 5, 6], 'C': [7, 8, None]}
df = pd.DataFrame(data)

# Display the original DataFrame
print("Original DataFrame:")
print(df)

# Replace the value in the second row and third column with the string "your_string"
df.at[2, 'C'] = 'your_string'
# or
# df.loc[1, 'C'] = 'your_string'

# Display the updated DataFrame
print("\nUpdated DataFrame:")
print(df)


Original DataFrame:
   A  B    C
0  1  4  7.0
1  2  5  8.0
2  3  6  NaN

Updated DataFrame:
   A  B            C
0  1  4          7.0
1  2  5          8.0
2  3  6  your_string


In [19]:
sc=pd.concat([result_dff,result_df2])
sc_1=sc.drop_duplicates()
new_column=['A','B','C']
sc_1.columns=new_column
sc_1.at[7, 'C'] = 'camphor'
sc_1

<ipython-input-19-77648a6f16c4>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sc_1.at[7, 'C'] = 'camphor'


,A,B,C
0,aldehidic,aldehydic,None
1,almond,almondy,None
2,balsamic,basalmic,balsam
3,buttery,butter,None
4,camphoraceous,camphoreous,None
5,caramelic,caramellic,caramel
6,citrus,citrusy,None
7,coco,cocoa,camphor
8,ehtereal,ethereal,None
9,floral,foral,None


In [20]:
result = pd.melt(sc_1, value_vars=['A', 'B', 'C'])

# Rename the columns (optional)
result.columns = ['name','Combined_Column']

# Display the original DataFrame
print("Original DataFrame:")
print(sc_1)

# Display the result with the combined column
print("\nDataFrame with Combined Column:")
print(result)


Original DataFrame:
                A            B        C
0       aldehidic    aldehydic     None
1          almond      almondy     None
2        balsamic     basalmic   balsam
3         buttery       butter     None
4   camphoraceous  camphoreous     None
5       caramelic   caramellic  caramel
6          citrus      citrusy     None
7            coco        cocoa  camphor
8        ehtereal     ethereal     None
9          floral        foral     None
10          fruit       fruity     None
11          gassy       grassy     None
12          green       green,     None
13         herbal       herba-     None
14         jasmin      jasmine     None
15        leather     leathery     None
16          lilac       lillac     None
17           mint        minty     None
18       mushroom    mushroomy     None
19           musk        musky     None
20           must        musty     None
21          peach       peachy     None
22         pepper      peppery     None
23           pine   

In [21]:
result_a=result['Combined_Column'].sort_values()
result_a=pd.DataFrame(result_a)
result_b=result_a.dropna()
name=result_b.iloc[:,0].tolist()
name

['alcohol',
 'alcoholic',
 'aldehidic',
 'aldehydic',
 'almond',
 'almondy',
 'anise',
 'anisic',
 'balsam',
 'balsam',
 'balsamic',
 'balsamic',
 'basalmic',
 'beef',
 'beefy',
 'butter',
 'buttery',
 'camphor',
 'camphor',
 'camphoraceous',
 'camphoreous',
 'camphoreous',
 'caramel',
 'caramelic',
 'caramellic',
 'cheese',
 'cheesy',
 'citrus',
 'citrusy',
 'coco',
 'cocoa',
 'coumarin',
 'coumarinic',
 'ehtereal',
 'ethereal',
 'floral',
 'foral',
 'fruit',
 'fruity',
 'garlic',
 'garlic)',
 'gassy',
 'grassy',
 'green',
 'green,',
 'herba-',
 'herbal',
 'jasmin',
 'jasmine',
 'leather',
 'leathery',
 'lilac',
 'lillac',
 'mint',
 'minty',
 'mushroom',
 'mushroomy',
 'musk',
 'musky',
 'must',
 'musty',
 'peach',
 'peachy',
 'pepper',
 'peppery',
 'pine',
 'piney',
 'root',
 'rooty',
 'sulfuraceous',
 'sulfurous',
 'vanillin',
 'vanlilin',
 'wine-like',
 'winelike',
 'wood',
 'woody']

## combine all data with same name labels

In [22]:
label_same=label_all.loc[:,name]
label_same

,alcohol,alcoholic,aldehidic,aldehydic,almond,almondy,anise,anisic,balsam,balsam,...,root,rooty,sulfuraceous,sulfurous,vanillin,vanlilin,wine-like,winelike,wood,woody
0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5251,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
5252,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
5253,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
5254,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


## Unique label

In [23]:
label_3=label_all.drop(columns =name)
label_3

,acid,ambre,animal,apple,apricot,aromatic,aromatique,banana,berry,brandy,...,"alliaceous (onion, garlic)",blossom,raspberry,fennel,iris,corn,mango,lavender,beer,alliaceous (onion
0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0,0,0,0.0,0.0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0,0,0,0.0,0.0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0,0,0,0.0,0.0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0,0,0,0.0,0.0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5251,0.0,0.0,0,0,0,0.0,0.0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5252,0.0,0.0,0,0,0,0.0,0.0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5253,0.0,0.0,0,0,0,0.0,0.0,0,0,0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
5254,0.0,0.0,1,0,0,0.0,0.0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Summrize in excel and save as same_all.xlsx

In [24]:
# pair with two name
label_sa=pd.read_excel('/content/same_alla.xlsx') #C:\wq\cg\capsuleml\dataset\data_analysis\common\update17jan
label_sa=label_sa.drop(columns=['Unnamed: 0'])
label_sa

,A,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,C
0,alcohol,NaN,recove,anise,NaN,NaN,NaN,1.0,balsam
1,alcoholic,NaN,NaN,anisic,NaN,NaN,NaN,2.0,balsamic
2,aldehidic,NaN,coco replace by cocoa,NaN,NaN,NaN,NaN,3.0,basalmic
3,aldehydic,NaN,NaN,NaN,NaN,NaN,NaN,4.0,camphor
4,almond,NaN,NaN,NaN,NaN,NaN,NaN,5.0,camphoraceous
...,...,...,...,...,...,...,...,...,...
63,gasoline,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
64,rose,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
65,rosy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
66,milky,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
# pair with three name
label_s3=pd.read_excel('/content/same_all_3.xlsx') #C:\wq\cg\capsuleml\dataset\data_analysis\common\update17jan
label_s3=label_s3.drop(columns=['Unnamed: 0'])
label_s3

,C
0,balsam
1,balsamic
2,basalmic
3,camphor
4,camphoraceous
5,camphoreous
6,caramel
7,caramelic
8,caramellic
9,herba-


In [26]:
import pandas as pd

# Sample DataFrame
data = {'Column': ['abcXyz', 'aBcDef', 'lmnOpq', 'AbCdEf']}
df = pd.DataFrame(data)

# Function to get the index of the first capital letter in a string
def get_first_capital_index(s):
    for i, char in enumerate(s):
        if char.isupper():
            return i
    return len(s)

# Apply the function to get the first capital index for each value in the column
df['FirstCapitalIndex'] = df['Column'].apply(get_first_capital_index)

# Sort the DataFrame based on the 'FirstCapitalIndex' column
df_sorted = df.sort_values(by='FirstCapitalIndex')

# Drop the temporary column used for sorting
df_sorted = df_sorted.drop(columns=['FirstCapitalIndex'])

# Display the sorted DataFrame
print(df_sorted)


   Column
3  AbCdEf
1  aBcDef
0  abcXyz
2  lmnOpq


In [27]:
name_a=label_sa['A'].values.tolist()
name_b=label_s3['C'].values.tolist()

In [ ]:
#name_a=label_sa.iloc[:,0].tolist()
#name_b=label_sa.iloc[:,1].tolist()

In [28]:
label_same1=label_all.loc[:,name_a]
label_same2=label_all.loc[:,name_b]
#label_same3=label_all.loc[:,name_3]
#label_same4=label.loc[:,['winey']]
label_same1

,alcohol,alcoholic,aldehidic,aldehydic,almond,almondy,beef,beefy,butter,buttery,...,aromatic,aromatique,orange,orange-blossom,gas,gasoline,rose,rosy,milky,dairy
0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0,0.0,0.0,0.0,0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0,0.0,0.0,0.0,0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0,0.0,0.0,0.0,0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0,0.0,0.0,0.0,0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0,0.0,0.0,0.0,0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5251,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0,0.0,0.0,0.0,0,0.0,0.0,0.0
5252,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0,0.0,0.0,0.0,0,0.0,0.0,0.0
5253,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0,0.0,0.0,0.0,1,0.0,0.0,0.0
5254,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0,0.0,0.0,0.0,0,0.0,0.0,0.0


In [ ]:
label_same2.to_csv('s3.csv')

### Unique label

In [29]:
#label_all=pd.read_csv('/content/drive/MyDrive/pgml/Data analysis/common/update17janlabel_all.csv')
label_n_1=label_all.drop(columns =name_a)
label_n_2=label_n_1.drop(columns =name_b)
label_n_2

,acid,ambre,animal,anisic,apple,apricot,banana,berry,brandy,creamy,...,sage,quince,blossom,raspberry,fennel,iris,corn,mango,lavender,beer
0,0.0,0.0,0,0.0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0,0.0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0,0.0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0,0.0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0,0.0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5251,0.0,0.0,0,0.0,0,0,0,0,0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5252,0.0,0.0,0,0.0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5253,0.0,0.0,0,0.0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
5254,0.0,0.0,1,0.0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [30]:
label_n_3=label_n_2.loc[:, (label_n_2 != 0).any(axis=0)]
label_n_3

,acid,ambre,animal,anisic,apple,apricot,banana,berry,brandy,creamy,...,sage,quince,blossom,raspberry,fennel,iris,corn,mango,lavender,beer
0,0.0,0.0,0,0.0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0,0.0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0,0.0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0,0.0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0,0.0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5251,0.0,0.0,0,0.0,0,0,0,0,0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5252,0.0,0.0,0,0.0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5253,0.0,0.0,0,0.0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
5254,0.0,0.0,1,0.0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [31]:
label_n_3=label_n_3.drop(columns=['woney'])

In [33]:
label_n_3.to_csv('unique_label.csv')

### combine same core meaning

In [32]:
label_sac=pd.concat([label_same1,label_same2],axis=1)
label_sac.to_csv('s.csv')
label_sac

,alcohol,alcoholic,aldehidic,aldehydic,almond,almondy,beef,beefy,butter,buttery,...,camphoreous,caramel,caramelic,caramellic,herba-,herbal,herbaceous,wine-like,winelike,winey
0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5251,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5252,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5253,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5254,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [34]:
import pandas as pd

# Sample DataFrame
data = {
    'column1': [1, 2, 3],
    'column2': [4, 5, 6],
    'column3': [7, 8, 9],
    'column4': [10, 11, 12]
}

df = pd.DataFrame(data)


# Get the list of column names
columns = df.columns

# Create new columns by adding every two adjacent columns
for i in range(0, len(columns)-1, 2):
    new_column_name = f"{columns[i]}_a"
    df[new_column_name] = df[columns[i]] + df[columns[i+1]]

# Display the modified DataFrame
print("\nModified DataFrame:")
print(df)



Modified DataFrame:
   column1  column2  column3  column4  column1_a  column3_a
0        1        4        7       10          5         17
1        2        5        8       11          7         19
2        3        6        9       12          9         21


### Two items pair

In [35]:
# Get the list of column names
columns = label_same1.columns

# Create new columns by adding every two adjacent columns
for i in range(0, len(columns)-1, 2):
    new_column_name = f"{columns[i]}_a"
    label_same1[new_column_name] = label_same1[columns[i]] + label_same1[columns[i+1]]
label_same1

,alcohol,alcoholic,aldehidic,aldehydic,almond,almondy,beef,beefy,butter,buttery,...,root_a,sulfuraceous_a,vanillin_a,wood_a,alliaceous (onion_a,aromatic_a,orange_a,gas_a,rose_a,milky_a
0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5251,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5252,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5253,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
5254,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [36]:
label_same1a=label_same1.iloc[:,-34:]
label_same1a

,alcohol_a,aldehidic_a,almond_a,beef_a,butter_a,cheese_a,citrus_a,coco_a,coumarin_a,ehtereal_a,...,root_a,sulfuraceous_a,vanillin_a,wood_a,alliaceous (onion_a,aromatic_a,orange_a,gas_a,rose_a,milky_a
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5251,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5252,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5253,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
5254,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [37]:
label_same1a.replace(2, 1, inplace=True)  # replace all value==2 to 1

In [38]:
label_same1a.to_csv('sa1.csv')

### Three items pair

In [39]:
# Get the list of column names
columns = label_same2.columns

# Create new columns by adding every two adjacent columns
for i in range(0, len(columns)-1, 3):
    new_column_name = f"{columns[i]}_b"
    label_same2[new_column_name] = label_same2[columns[i]] + label_same2[columns[i+1]]+ label_same2[columns[i+2]]
label_same2

,balsam,balsamic,basalmic,camphor,camphoraceous,camphoreous,caramel,caramelic,caramellic,herba-,herbal,herbaceous,wine-like,winelike,winey,balsam_b,camphor_b,caramel_b,herba-_b,wine-like_b
0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5251,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5252,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5253,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5254,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [40]:
label_same2a=label_same2.iloc[:,-5:]
label_same2a.replace(2, 1, inplace=True)  # replace all value==2 to 1
label_same2a.replace(3, 1, inplace=True)  # replace all value==2 to 1
label_same2a
label_same2a.to_csv('sa2.csv')

## The label data of common molecular manually combine togther

In [41]:
all_m_feature=pd.read_csv('/content/drive/MyDrive/pgml/Data analysis/common/update17jan/all_data_feature_c.csv')
all_m_feature

<ipython-input-41-fcbe9ef9e51d>:1: DtypeWarning: Columns (2085,2086,2087,2088,2089,2090,2091,2092,2093,2094,2095,2096,2156,2157,2158,2159,2160,2165,2166,2167,2168,2169,2174,2175,2176,2177,2178,2183,2184,2185,2186,2187,2192,2193,2194,2195,2196,2201,2202,2203,2204,2205,2264,2265,2266,2267,2268,2273,2274,2275,2276,2277,2282,2283,2284,2285,2286,2291,2292,2293,2294,2295,2300,2301,2302,2303,2304,2309,2310,2311,2312,2313,2317,2318,2319,2320,2321,2325,2326,2327,2328,2329,2333,2334,2335,2336,2337,2341,2342,2343,2344,2345,2349,2350,2351,2352,2353,2357,2358,2359,2360,2361,2365,2366,2367,2368,2369,2373,2374,2375,2376,2377,2381,2382,2383,2384,2385,2389,2390,2391,2392,2393,2397,2398,2399,2400,2401,2405,2406,2407,2408,2409,2413,2414,2415,2416,2417,2418,2419,2420,2421,2422,2423,2424,2425,2426,2427,2428,2429,2430,2431,2432,2433,2434,2435,2436,2437,2438,2439,2440,2441,2442,2443,2444,2445,2446,2447,2448,2449,2450,2451,2452,2453,2454,2455,2456,2457,2458,2459,2460,2461,2462,2463,2464,2465,2466,2467,2468,24

,CID,MaxEStateIndex,MinEStateIndex,MaxAbsEStateIndex,MinAbsEStateIndex,qed,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,...,TSRW10,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2,PERSISTENCE_ENTROPY_DIM_0,PERSISTENCE_ENTROPY_DIM_1,PERSISTENCE_ENTROPY_DIM_2
0,10569.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.904445,2.729808,0.693058
1,101602.0,13.723676,-5.416264,13.723676,2.287573,0.760294,302.458,272.218,302.224580,122.0,...,57.302060,943.0,45.0,124.0,154.0,8.791666666666666,4.638889,3.947052,2.529469,0.485423
2,22235151.0,9.745935,-5.695194,9.745935,2.728143,0.693521,290.491,256.219,290.260966,120.0,...,56.028098,834.0,42.0,118.0,146.0,7.930555555555555,4.513889,3.424074,1.140838,0.000000
3,68074566.0,7.945269,-4.130607,7.945269,0.916573,0.509873,210.273,192.129,210.125594,84.0,...,45.147991,458.0,15.0,64.0,67.0,4.972222222222222,3.666667,3.067019,1.568200,0.000000
4,31361.0,7.560417,-3.862917,7.560417,3.648895,0.495006,130.187,116.075,130.099380,54.0,...,36.195345,88.0,10.0,38.0,39.0,2.861111111111111,2.166667,3.141797,1.336667,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5292,6287.0,11.316319,-3.852546,11.316319,0.865463,0.536977,117.148,106.060,117.078979,48.0,...,35.071670,65.0,8.0,32.0,33.0,5.333333,1.888889,2.853617,1.355357,0.000000
5293,6781.0,12.358586,-3.629951,12.358586,1.086318,0.731363,222.240,208.128,222.089209,86.0,...,47.511782,446.0,22.0,72.0,81.0,6.444444,3.916667,3.345647,1.552718,0.000000
5294,8615.0,12.690049,-3.735398,12.690049,0.325212,0.661030,241.290,226.170,241.110279,92.0,...,50.261825,702.0,23.0,86.0,96.0,5.444444,4.138889,3.435885,1.277218,0.000000
5295,31268.0,7.192708,-3.078125,7.192708,0.291667,0.434794,71.123,62.051,71.073499,30.0,...,41.004802,15.0,0.0,20.0,20.0,1.25,1.250000,2.553369,0.244144,0.000000


In [42]:
label_all=pd.read_csv('/content/drive/MyDrive/pgml/Data analysis/common/update17jan/label_all.csv')
label_all

,CID,acid,aldehidic,aldehydic,almond,almondy,ambre,animal,anisic,apple,...,raspberry,fennel,iris,corn,balsam,mango,lavender,beer,alliaceous (onion,garlic)
0,10569.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,101602.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,22235151.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,68074566.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,31361.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5251,6287.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5252,6781.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5253,8615.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5254,31268.0,0.0,0.0,0.0,0,0.0,0.0,1,0.0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [43]:
label_f3=pd.read_csv('/content/same_label_com.csv') # C:\wq\cg\capsuleml\dataset\data_analysis\common\update17jan\same
label_f3=label_f3.drop(columns=['Unnamed: 0'])
label_f3

,alcohol,aldehidic,almond,beefy,butter,cheesey,citrus,coconut,coumarin,ehtereal,...,aromatic,orange,gas,rose,milky,balsamic,camphoreous,caramelic,herbal,winey
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5251,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5252,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5253,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
5254,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [44]:
com_label_f=pd.concat([label_all['CID'],label_f3],axis=1)
#com_label_f=com_label_f.drop(columns='Unnamed: 0')
com_label_f

,CID,alcohol,aldehidic,almond,beefy,butter,cheesey,citrus,coconut,coumarin,...,aromatic,orange,gas,rose,milky,balsamic,camphoreous,caramelic,herbal,winey
0,10569.0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,101602.0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,22235151.0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,68074566.0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,31361.0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5251,6287.0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5252,6781.0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5253,8615.0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
5254,31268.0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [45]:
com_label_f1=com_label_f.loc[:, (com_label_f != 0).any(axis=0)]

In [46]:
## combine common and unique labels
label_all_clean=pd.concat([com_label_f1,label_n_3],axis =1)
label_all_clean

,CID,alcohol,aldehidic,almond,beefy,butter,cheesey,citrus,coconut,coumarin,...,sage,quince,blossom,raspberry,fennel,iris,corn,mango,lavender,beer
0,10569.0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,101602.0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,22235151.0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,68074566.0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,31361.0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5251,6287.0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5252,6781.0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5253,8615.0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
5254,31268.0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [47]:
label_all_clean.loc[:, (label_all_clean != 0).any(axis=0)]

,CID,alcohol,aldehidic,almond,beefy,butter,cheesey,citrus,coconut,coumarin,...,sage,quince,blossom,raspberry,fennel,iris,corn,mango,lavender,beer
0,10569.0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,101602.0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,22235151.0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,68074566.0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,31361.0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5251,6287.0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5252,6781.0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5253,8615.0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
5254,31268.0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [48]:
label_all_clean.to_csv('label_all_clean.csv',index=False)